# Phase 1: Document Classification & Generic Metadata Model

The previous notebook (02) established a working extraction pipeline for 
financial documents. 

This notebook refactors the data model towards a more generic approach:
- A base class `DocumentMetadata` for all document types
- Specialized subclasses (e.g. `FinancialDocument`, `Contract`) that extend the base
- A two-step pipeline: first classify the document type, then extract 
  with the appropriate schema

In [27]:
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pathlib import Path
from typing import Optional, List, Any
from datetime import timedelta
import dateparser
import json
import os
import pdfplumber

In [28]:
# Configuration
load_dotenv()
client = OpenAI(api_key=os.getenv("LM_STUDIO_API_KEY"),
base_url=os.getenv("LM_STUDIO_BASE_URL"))
LLM_MODEL = str(os.getenv("LM_STUDIO_MODEL"))
if not LLM_MODEL:
    raise ValueError("LM_STUDIO_MODEL not configured. Check your .env file.")

data_dir = Path.cwd().parent / "data" / "samples"

In [29]:
# DataModel
class DocumentMetaData(BaseModel):
    """Base class for all document types"""
    document_type: str
    language: Optional[str]
    summary: Optional[str]
    tags: Optional[List[str]]
    date: Optional[str]
    issuer: Optional[str]
    recipient: Optional[List[str]]
    document_id: Optional[str]

class DocFile(BaseModel):
    """Any file located in the file system"""
    model_config = {"arbitrary_types_allowed": True}
    file_name: str
    suffix: str
    path: Path

class FinancialDocument(DocumentMetaData):
    """Financial documents: invoices, receipts, tax documents"""
    amount: Optional[float]
    currency: Optional[str]
    status: Optional[str]
    due_date: Optional[str] = Field(description="Calendar Due Date, e.g. 2026-04-20 ")
    customer_id: Optional[str] = Field(description="Customer number, e.g. Kd-Nr. 1010")
    payment_terms_days: Optional[int]
    iban: Optional[str]
    bic: Optional[str]

class LegalDocument(DocumentMetaData):
    """Contracts and agreements"""
    valid_from: Optional[str]
    valid_until: Optional[str]
    cancellation_period_days: Optional[int]
    parties: Optional[List[str]]

class EmailDocument(DocumentMetaData):
    """Email documents"""
    subject: Optional[str]
    cc: Optional[List[str]]
    attachments: Optional[List[str]]

In [30]:
# File Crawler
def crawl_directory(path: Path, files: list[DocFile] | None = None) -> dict[str, Any]:
    if files is None:
        files = []
    for elem in path.iterdir():
        if elem.is_file():
            file = DocFile(file_name=elem.name, suffix=str.lower(elem.suffix), path=elem)
            files.append(file)
        if elem.is_dir():
            crawl_directory(elem, files)
    content = {
        "files": files,
        "num_files": len(files)
    }
    return content

In [31]:
# Schema mapping
SCHEMA_MAP = {
    "FinancialDocument": FinancialDocument,
    "LegalDocument": LegalDocument,
    "EmailDocument": EmailDocument,
}

In [32]:
# Few Shot Example 1
example1 = """Musterfirma GmbH · Hauptstraße 1 · 80331 München

Familie Müller
Max Müller
Musterweg 5
12345 Berlin

Rechnung Nr.: RE-2024-001    Kd-Nr.: 4711
Datum: 15.3.2024

Pos. 1: Beratungsleistung März 2024    250,00 €
Gesamt: 250,00 €

Zahlbar innerhalb von 14 Tagen.
IBAN: DE12 3456 7890 1234 5678 90
BIC: MUELDE12
"""

example_reply1 = json.dumps({
    "document_type": "Invoice",
    "language": "German",
    "summary": "Rechnung der Musterfirma GmbH an Max Müller für Beratungsleistungen.",
    "tags": ["Beratung"],
    "date": "15.3.2024",
    "issuer": "Musterfirma GmbH",
    "recipient": ["Max Müller"],
    "document_id": "RE-2024-001",
    "amount": 250.00,
    "currency": "EUR",
    "status": None,
    "due_date": None,
    "customer_id": "4711",
    "payment_terms_days": 14,
    "iban": "DE12 3456 7890 1234 5678 90",
    "bic": "MUELDE12"
})

# Few Shot Example 2
example2 = """TechCorp Ltd. · 10 Business Park · London EC1A 1BB

Global Solutions Inc.
Attn: Accounts Payable
500 Market Street
New York, NY 10001

Invoice No.: TC-2024-042    Date: 2024-06-01
Due Date: 2024-06-30

Item 1: Software License Q3 2024    1,200.00 EUR
Total: 1,200.00 EUR

Status: OPEN
IBAN: GB29 NWBK 6016 1331 9268 19
BIC: NWBKGB2L
"""

example_reply2 = json.dumps({
    "document_type": "Invoice",
    "language": "English",
    "summary": "Invoice from TechCorp Ltd. to Global Solutions Inc. for a software license.",
    "tags": ["Software", "License"],
    "date": "2024-06-01",
    "issuer": "TechCorp Ltd.",
    "recipient": ["Global Solutions Inc."],
    "document_id": "TC-2024-042",
    "amount": 1200.00,
    "currency": "EUR",
    "status": "OPEN",
    "due_date": "2024-06-30",
    "customer_id": None,
    "payment_terms_days": None,
    "iban": "GB29 NWBK 6016 1331 9268 19",
    "bic": "NWBKGB2L"
})

In [33]:
# Get the files
data_dir = Path.cwd().parents[0] / "data"
d = crawl_directory(data_dir)
for f in d['files']:
    doc = []
    if not f.suffix == '.pdf':
        print(f"{f.suffix} Files cannot be processed yet.")
        continue
    with pdfplumber.open(f.path) as pdf:
        for i, page in enumerate(pdf.pages):
            page_num = i +1
            page_text = page.extract_text()
            doc.append({'page_num': page_num, 'text': page_text if page_text else ""})

    # Step 1: Categorize the document
    completion = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system",
            "content": """Identität:
            Du bist ein Spezialist für Dokumente.
            
            Aufgabe:
            Bestimme die Kategorie des Dokuments.
            Beispiel:
            Q: Rechnungen, Invoices A: FinancialDocument
            Q: Verträge, Contracts A: LegalDocument
            Q: Emails A: EmailDocument
            Antworte in nur einem Wort mit dem englischen Begriff."""},
            {"role": "user",
            "content": f"{doc}"}
        ]
    )
    content = completion.choices[0].message.content
    if not content:
        raise ValueError("Model returned empty response")
    
    doc_class = content.strip().split()[0]  # nur erstes Wort

    # Dem Dictionary hinzufügen
    doc_info = {
        "file": f,
        "pages": doc,
        "doc_class": doc_class,
        "schema": SCHEMA_MAP.get(doc_class, DocumentMetaData)  # Fallback auf Basisklasse
    }
    # Step 2: Extraction mit hlife des Schemas
    completion = client.chat.completions.create(
        model=LLM_MODEL,  # exakter Name aus LM Studio
        messages=[
            {"role": "system", "content": """
            Du bist ein Buchhaltungs- und Ablage-Assistent.

    Aufgabe:
    Extrahiere die angeforderten Felder ausschließlich aus dem vorliegenden Dokument.

    Felddefinitionen:
    - issuer: Nur der Name, keine Adresse oder Kontaktdaten
    - recipient: Nur der Name, keine Adresse
    - doc_lang: Englische Bezeichnung der Sprache, in der das Dokument verfasst ist                  
    - document_type: Englische Bezeichnung des Dokument-Typs: "Rechnung" -> "Invoice"         
    - document_id: Nur die Rechnungsnummer. Erkennbar an Begriffen wie "Rech.-Nr.", "Rechnungsnummer", "Invoice No." Beispiel: GB-2026/001
    - customer_id: Nur die Kundennummer. Erkennbar an Begriffen wie "Kd-Nr.", "Kundennummer", "Customer No." Beispiel: 1010
    - date: Ausstellungsdatum des Dokuments. Erkennbar an Begriffen wie "Datum:", "Rechnungsdatum", "Date".
    - due_date: NUR ein Datum das WÖRTLICH im Dokument steht nach Begriffen wie 
    "Fällig bis", "Due date", "Zahlbar bis". 
    Wenn kein solches Datum im Text vorkommt → zwingend null.
    Berechnungen sind VERBOTEN. Zahlungsfristen wie "14 Tage" sind KEIN Fälligkeitsdatum.
    - iban: Bankverbindung IBAN. Erkennbar an "IBAN:" gefolgt von einem Kontocode.
    - bic: BIC/SWIFT Code der Bank. Erkennbar an "BIC:" oder "SWIFT:".
    - summary: Ein Satz der den Inhalt des Dokuments beschreibt, nicht die Aufgabe des Assistenten                           
            
    Regeln:
    - Extrahiere nur Informationen die explizit im Dokument stehen
    - Erfinde oder schlussfolgere keine Werte
    - Wenn ein Feld nicht eindeutig erkennbar ist, gib null zurück
    - Der Status (z.B. open, paid, overdue) darf nur gesetzt werden wenn er 
    ausdrücklich im Dokument erwähnt wird
    - Beträge werden als Dezimalzahl mit zwei Nachkommastellen zurückgegeben"""},
    {"role": "user", "content": f"Dokument:\n {example1}"},
    {"role": "assistant", "content": f"{example_reply1}"},
    {"role": "user", "content": f"Dokument:\n {example2}"},
    {"role": "assistant", "content": f"{example_reply2}"},
    {"role": "user", "content": f"Dokument:\n {doc_info['pages']}"}],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": doc_info['doc_class'],
                "schema": doc_info['schema'].model_json_schema()
            }
        },
    )
    content = completion.choices[0].message.content
    if not content:
        raise ValueError("Model returned empty response")

    result = doc_info['schema'].model_validate_json(content)
    print(f"Raw due_date: {result.due_date}")

    for field in doc_info['schema'].model_fields:
        if getattr(result, field) == 'null':
            setattr(result, field, None)
    
    if result.language:
        result.language = result.language.capitalize()
    
    if result.due_date is None and result.date is not None and result.payment_terms_days is not None and result.language is not None:
        if result.language != "English":
            doc_date = dateparser.parse(result.date, languages=["de", "en"], settings={"DATE_ORDER": "DMY"})
        else:
            doc_date = dateparser.parse(result.date, languages=["de", "en"], settings={"DATE_ORDER": "YMD"})
        if doc_date:
            result.due_date = (doc_date + timedelta(days=result.payment_terms_days)).strftime("%Y-%m-%d")
        else:    
            print(f"⚠️ Datumsformat nicht erkannt: {result.date}")
    
    print(f"\n=== {doc_info['file']} ===")
    doc_info['fields'] = result 
    print(doc_info['fields'])   
    

    

Raw due_date: None

=== file_name='amazon.pdf' suffix='.pdf' path=WindowsPath('d:/Coding/Projects/content-aware-fileserver/data/samples/amazon.pdf') ===
document_type='Invoice' language='German' summary='Rechnung von Amazon EU S.à r.l., Niederlassung Deutschland an JESCO ALEXANDER WURM für ein Generatives KI-System.' tags=['KI-Engineering'] date='2026-02-18' issuer='Amazon EU S.à r.l., Niederlassung Deutschland' recipient=['JESCO ALEXANDER WURM'] document_id='DE6LLJ5FAEUI' amount=54.9 currency='EUR' status=None due_date=None customer_id=None payment_terms_days=None iban=None bic=None
.docx Files cannot be processed yet.
Raw due_date: None

=== file_name='GB.pdf' suffix='.pdf' path=WindowsPath('d:/Coding/Projects/content-aware-fileserver/data/samples/GB.pdf') ===
document_type='Invoice' language='German' summary='Rechnung des Gut Beraten Lohnsteuerhilfe e.V. an Julia und Jesco Wurm für den Jahresbeitrag 2026.' tags=['Jahresbeitrag', 'Einkommensteuererklärung'] date='3.1.2026' issuer='Gu

## Key Learning: Few-Shot Prompting

Zero-shot prompt engineering with explicit rules showed diminishing returns —
fixing one field tended to regress another.

Adding two synthetic few-shot examples resolved all remaining issues in one step:
- `recipient` extraction became consistent across all documents
- `language` detection stabilized
- `summary` quality improved significantly

Takeaway: For structured extraction tasks, few-shot examples are more effective
than increasingly complex rule sets.